# Reproducing Higgins et al. (2017), Figure 7A and 7B — β-VAE on dSprites

**A complete, independently written PyTorch reproduction for Kaggle (two NVIDIA T4 GPUs).**

This notebook **trains a β-VAE with β=4 and a standard VAE with β=1 from scratch**, performs full-dataset posterior inference, measures factor-conditioned latent responses, and creates independent panels corresponding to Figure 7A and 7B. Neither the paper's plotted measurements nor pre-trained weights are used. No existing implementation was consulted.

**Run on Kaggle:** Settings → Accelerator → **GPU T4 x2**. Turn Internet **on** for the initial data download, or attach the official dSprites `.npz` as an Input with Internet off. Run cells in order. The notebook writes its own standalone worker script to `/kaggle/working`, so it requires no additional repository files. All outputs go under `/kaggle/working/beta_vae_fig7/` (or `./beta_vae_fig7/` outside Kaggle).

**Experiment commitments:** one seed per β; two independent GPUs; every one of 737,280 images; 64×64 pixels; 10 Gaussian latents; paper's 1200-unit fully connected network; Adagrad 0.01; 128 images/minibatch; 40 epochs; float32 everywhere; no pretrained weights; no mixed precision; no data subsampling. The 40-epoch budget is an explicit replication choice, **not** a training duration supplied by the paper.

**Source basis:** Higgins et al., *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*, §2, §4.2, Fig. 7, Table 1, Appendix A.6; Assignment 2 §1.1 item 2 and §4. A single trained seed cannot establish statistical robustness or a distribution of outcomes.

## 1. Method, reproducibility, and the time calculation

The model maximizes 

\[\mathcal L_\beta(x)=\mathbb E_{q_\phi(z\mid x)}\log p_\theta(x\mid z)-\beta D_{KL}(q_\phi(z\mid x)\|\mathcal N(0,I)).\]

With $q_\phi(z\mid x)=\mathcal N(\mu_\phi(x),\operatorname{diag}(e^{\ell_\phi(x)}))$, $z=\mu+e^{\ell/2}\odot\epsilon$ and $\epsilon\sim\mathcal N(0,I)$, the **negative** objective is

\[\underbrace{\sum_{p=1}^{4096}\operatorname{BCEWithLogits}(a_p,x_p)}_{\text{sum over pixels}}+\beta\underbrace{\frac12\sum_{j=1}^{10}\left(\mu_j^2+e^{\ell_j}-1-\ell_j\right)}_{\text{sum over latents}},\]

**then averaged over examples**. The reductions are essential to preserve the stated β. For β>1, this β-weighted objective is a (possibly looser) lower bound on log evidence; for 0≤β<1 it is not guaranteed to be a lower bound. The KL penalty controls information capacity, which can trade off against reconstruction fidelity; larger β need not always yield better disentanglement.

### 40 epochs: auditable estimate, not a benchmark result

- Training observations: $3\times6\times40\times32\times32=737{,}280$.
- Batch size: 128; **exactly 5,760 gradient steps per epoch**, 230,400 steps per model over 40 epochs.
- Parameters: encoder $(4096\cdot1200+1200)+(1200\cdot1200+1200)+2(1200\cdot10+10)$; decoder $(10\cdot1200+1200)+2(1200\cdot1200+1200)+(1200\cdot4096+4096)$, about **14.2 million** total.
- Approximate forward-plus-backward multiply/add computation: $\approx6P B\approx10.9$ GFLOPs/step, plus elementwise activations, Adagrad, data gathering, and Python overhead.
- Two models run **at the same time**, one per T4, so wall time is governed by the slower GPU, not the sum of both GPU times.
- **Conditional wall-time scenarios** for 40 epochs (not measured on Kaggle): 50 ms/step → **3 h 12 min**; 65 ms/step → **4 h 10 min**; 80 ms/step → **5 h 7 min**; 95 ms/step → **6 h 5 min**, **before** data download, initialization, analysis, and plotting. Effective throughput on Kaggle must be checked in the provided preflight. If the measured time exceeds your allowance, **do not silently change the epoch count**: stop and deliberately choose whether to accept extra time or a documented partial run.

A short optional **separately labeled** benchmarking cell, with fresh randomly initialized models and no training state carried forward, measures the local step time before training. Actual training always begins from its declared seed and runs for exactly 40 epochs unless stopped, in which case partial checkpoints are labeled as such. Checkpoints are written after every epoch for the first three epochs, then every five epochs, and at the final epoch. A run interrupted between checkpoints loses only the unsaved portion. Restarting resumes full optimizer and RNG state.

**No scientific cherry-picking:** both models use the same architecture, seed, initialization, batch order, and initial random streams (though training diverges as β changes). There is no hyperparameter search or stopping based on the desired visual appearance.

In [ ]:
import os, sys, json, time, math, subprocess, shutil, pathlib, textwrap, hashlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle/working').exists() else '.').resolve()
OUT = ROOT / 'beta_vae_fig7'
OUT.mkdir(parents=True, exist_ok=True)
DATA_NAME = 'dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'
DATA_PATH = OUT / DATA_NAME
EPOCHS = 40     # approved fixed epoch target; do not adjust automatically
BATCH_SIZE = 128
SEED = 20261020
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, (
    'Select Kaggle GPU T4 x2 under Settings → Accelerator. Both GPUs are required for parallel execution.')
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
for i in range(2):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}; {torch.cuda.get_device_properties(i).total_memory/2**30:.2f} GiB')
print('Output directory:', OUT)

## 2. Acquire and validate the original dSprites archive

Default: download **only the official data file** from the dSprites dataset project, not code. If Kaggle Internet is disabled, attach the official `.npz` to the notebook Inputs; the cell automatically detects that attachment. For manual control, set `OVERRIDE_DATA_PATH` to its path. If a local, previously verified archive exists, it is reused. The validation cell loads the complete array, checks shape, dtype, pixel domain, complete balanced factor grid, and canonical enumeration. The factor labels are **never used to train** either model.

In [ ]:
import urllib.request
OVERRIDE_DATA_PATH = None    # e.g. '/kaggle/input/dsprites/<filename>.npz'
DATA_URL = ('https://github.com/deepmind/dsprites-dataset/raw/master/' + DATA_NAME)

def validate_archive(path):
    with np.load(path, allow_pickle=False) as d:
        if 'imgs' not in d or 'latents_classes' not in d:
            raise ValueError('Missing imgs or latents_classes: this is not the expected dSprites archive')
        imgs, fac = d['imgs'], d['latents_classes']
        assert imgs.shape == (737280, 64, 64), imgs.shape
        assert fac.shape == (737280, 6), fac.shape
        assert imgs.dtype in (np.dtype('uint8'), np.dtype('bool')), imgs.dtype
        assert np.all((imgs == 0) | (imgs == 1)), 'Images must be binary'
        # Canonical dSprites axis order: color, shape, scale, orientation, posX, posY.
        sizes = np.array([1,3,6,40,32,32])
        assert np.array_equal(fac.min(0), np.zeros(6))
        assert np.array_equal(fac.max(0), sizes - 1)
        weights = np.array([737280, 245760, 40960, 1024, 32, 1])
        ids = (fac.astype(np.int64) * weights).sum(axis=1)
        assert np.array_equal(ids, np.arange(737280)), 'Factor labels are not in canonical Cartesian order'
    print('Validated complete 737,280-image dSprites grid:', path)
    return pathlib.Path(path)

if OVERRIDE_DATA_PATH:
    DATA_PATH = pathlib.Path(OVERRIDE_DATA_PATH)
else:
    attached = sorted(pathlib.Path('/kaggle/input').rglob(DATA_NAME)) if pathlib.Path('/kaggle/input').exists() else []
    if attached:
        DATA_PATH = attached[0]
    elif not DATA_PATH.exists():
        print('Downloading official dSprites data archive (requires Kaggle Internet on)...')
        partial = OUT / (DATA_NAME + '.download')
        try:
            urllib.request.urlretrieve(DATA_URL, partial)
            partial.replace(DATA_PATH)
        except Exception as e:
            partial.unlink(missing_ok=True)
            raise RuntimeError('Download failed. Enable Kaggle Internet, or attach the official dSprites .npz file as an Input.') from e
DATA_PATH = validate_archive(DATA_PATH)
print('Compressed archive size (GiB):', round(DATA_PATH.stat().st_size / 2**30, 3))

## 3. Training worker (automatically created from this notebook)

Jupyter-defined multiprocessing functions are unreliable with CUDA's spawn method. This cell writes an entirely self-contained, original worker script and launches **two ordinary Python processes**, each restricted to one different GPU via `CUDA_VISIBLE_DEVICES`. The workers share no model parameters. Both cache the full dataset on their own GPU in **uint8** form when it fits: 737,280 × 64 × 64 bytes ≈ 2.81 GiB each. Data are converted to float32 only for the current minibatch. If caching does not fit, the worker safely retains the complete uint8 dataset in **CPU memory** and copies every minibatch to the GPU. Neither path discards images.

In [ ]:
WORKER_SOURCE = "import os, sys, json, time, math, random, argparse, pathlib\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nN, D, Z = 737280, 4096, 10\n\ndef seed_everything(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.set_float32_matmul_precision('highest')\n\nclass BetaVAE(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.encoder = nn.Sequential(nn.Linear(4096,1200),nn.ReLU(),nn.Linear(1200,1200),nn.ReLU())\n        self.mu = nn.Linear(1200,10)\n        self.logvar = nn.Linear(1200,10)\n        self.decoder = nn.Sequential(nn.Linear(10,1200),nn.Tanh(),nn.Linear(1200,1200),nn.Tanh(),\n                                     nn.Linear(1200,1200),nn.Tanh(),nn.Linear(1200,4096))\n    def encode(self,x):\n        h = self.encoder(x.reshape(-1,D))\n        return self.mu(h), self.logvar(h)\n    def decode(self,z):return self.decoder(z)\n    def forward(self,x):\n        mu, lv = self.encode(x)\n        z = mu + torch.exp(0.5*lv)*torch.randn_like(mu)\n        return self.decode(z), mu, lv\n\ndef initialize(m):\n    if isinstance(m,nn.Linear):\n        bound = 1/math.sqrt(m.in_features)\n        nn.init.uniform_(m.weight,-bound,bound)\n        nn.init.uniform_(m.bias,-bound,bound)\n\ndef loss_parts(logits, x, mu, lv, beta):\n    reconstruction = F.binary_cross_entropy_with_logits(logits,x.reshape(-1,D),reduction='none').sum(dim=1)\n    kl = (-0.5*(1+lv-mu.square()-lv.exp())).sum(dim=1)\n    return (reconstruction + beta*kl).mean(), reconstruction.mean(), kl.mean()\n\ndef save_checkpoint(path, epoch, model, opt, shufgen, beta, seed, history, mode):\n    record = dict(epoch=epoch, model=model.state_dict(), optimizer=opt.state_dict(),\n                  shufgen_state=shufgen.get_state(), cpu_rng=torch.get_rng_state(),\n                  cuda_rng=torch.cuda.get_rng_state(), beta=beta, seed=seed,\n                  history=history, data_mode=mode, batch_size=128)\n    tmp = path.with_suffix('.tmp')\n    torch.save(record,tmp)\n    os.replace(tmp,path)\n\ndef main():\n    ap=argparse.ArgumentParser()\n    for key, typ in [('data',str),('out',str),('beta',float),('seed',int),('epochs',int),('batch',int)]:\n        ap.add_argument('--'+key, required=True, type=typ)\n    ap.add_argument('--benchmark',action='store_true')\n    args=ap.parse_args()\n    assert args.batch==128, 'Protocol fixed at batch=128'\n    assert torch.cuda.device_count()==1, 'Worker must see exactly one GPU'\n    device=torch.device('cuda:0'); seed_everything(args.seed)\n    out=pathlib.Path(args.out);out.mkdir(parents=True,exist_ok=True)\n    model=BetaVAE().to(device);model.apply(initialize)\n    optimizer=torch.optim.Adagrad(model.parameters(),lr=0.01,lr_decay=0,\n                                   weight_decay=0,initial_accumulator_value=0,eps=1e-10)\n    shufgen=torch.Generator(device=device).manual_seed(args.seed+1)\n    # Load original binary images, preserving their canonical Cartesian order.\n    print('Loading complete dSprites...',flush=True)\n    with np.load(args.data,allow_pickle=False) as d:\n        arr=d['imgs']\n        if arr.shape!=(N,64,64) or arr.dtype not in (np.uint8,np.bool_):\n            raise ValueError('Unexpected image data')\n        imgs=torch.from_numpy(np.ascontiguousarray(arr.reshape(N,D),dtype=np.uint8))\n    mode='cpu_uint8'\n    if torch.cuda.mem_get_info()[0] > imgs.numel()+2*1024**3:\n        try:\n            imgs=imgs.to(device)\n            mode='gpu_uint8'\n        except torch.cuda.OutOfMemoryError:\n            torch.cuda.empty_cache()\n            mode='cpu_uint8'\n    print('Image cache:',mode,'|',imgs.shape,'|',imgs.dtype,flush=True)\n    ckpt=out/'last.pt'; history=[]; start=0\n    if not args.benchmark and ckpt.exists():\n        try: previous=torch.load(ckpt,map_location=device,weights_only=False)\n        except TypeError:previous=torch.load(ckpt,map_location=device)\n        if previous['beta']!=args.beta or previous['seed']!=args.seed or previous['batch_size']!=args.batch:\n            raise RuntimeError('Checkpoint configuration mismatch')\n        model.load_state_dict(previous['model']);optimizer.load_state_dict(previous['optimizer'])\n        shufgen.set_state(previous['shufgen_state'])\n        torch.set_rng_state(previous['cpu_rng']);torch.cuda.set_rng_state(previous['cuda_rng'])\n        start=previous['epoch'];history=previous['history']\n        print('Resuming after epoch',start,flush=True)\n    if args.benchmark:\n        # Warm-up plus timed train steps on current hardware; does NOT save a checkpoint.\n        torch.cuda.synchronize()\n        durations=[]\n        for i in range(25):\n            ids=torch.randint(0,N,(args.batch,),device=device,generator=shufgen)\n            x=imgs[ids].to(device,dtype=torch.float32)\n            t=time.perf_counter()\n            optimizer.zero_grad(set_to_none=True)\n            logits,mu,lv=model(x);total,_,_=loss_parts(logits,x,mu,lv,args.beta)\n            total.backward();optimizer.step()\n            torch.cuda.synchronize()\n            if i>=5:durations.append(time.perf_counter()-t)\n        result=dict(beta=args.beta,seconds_per_step=float(np.median(durations)),\n                    estimated_train_hours=args.epochs*math.ceil(N/args.batch)*float(np.median(durations))/3600,\n                    cache_mode=mode,steps_measured=len(durations))\n        (out/'benchmark.json').write_text(json.dumps(result,indent=2))\n        print('BENCHMARK',json.dumps(result),flush=True)\n        return\n    for epoch in range(start,args.epochs):\n        model.train(); t0=time.perf_counter()\n        order=torch.randperm(N,device=device,generator=shufgen)\n        accum=np.zeros(3,dtype=np.float64)\n        for start_idx in range(0,N,args.batch):\n            ids=order[start_idx:start_idx+args.batch]\n            # GPU cache and CPU fallback preserve exact same input samples.\n            if mode=='cpu_uint8':\n                x=imgs[ids.cpu()].to(device=device,dtype=torch.float32)\n            else:\n                x=imgs[ids].to(dtype=torch.float32)\n            optimizer.zero_grad(set_to_none=True)\n            logits,mu,lv=model(x);total,rec,kl=loss_parts(logits,x,mu,lv,args.beta)\n            if not torch.isfinite(total):raise FloatingPointError('Nonfinite loss at epoch '+str(epoch+1))\n            total.backward();optimizer.step()\n            accum += np.array([total.item(),rec.item(),kl.item()])*len(ids)\n        elapsed=time.perf_counter()-t0\n        row=dict(epoch=epoch+1,objective=accum[0]/N,reconstruction_bce=accum[1]/N,\n                 kl=accum[2]/N,epoch_seconds=elapsed,beta=args.beta)\n        history.append(row)\n        (out/'history.json').write_text(json.dumps(history,indent=2))\n        print('EPOCH',json.dumps(row),flush=True)\n        if epoch+1<=3 or (epoch+1)%5==0 or epoch+1==args.epochs:\n            save_checkpoint(ckpt,epoch+1,model,optimizer,shufgen,args.beta,args.seed,history,mode)\n            print('CHECKPOINT',epoch+1,flush=True)\n    print('COMPLETE',args.epochs,flush=True)\n\nif __name__=='__main__':main()\n"
WORKER_PATH = OUT / 'beta_vae_worker.py'
WORKER_PATH.write_text(WORKER_SOURCE, encoding='utf-8')
print('Wrote original standalone training worker:', WORKER_PATH)

## 4. Mathematical and architecture unit tests (no training data consumed)

These tests catch two common silent reproduction errors: using mean-pixel BCE instead of sum-pixel BCE, and an incorrect Gaussian KL calculation. They also verify latent/decoder shape and gradient flow. The tests do not constitute a reproduction result.

In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('fig7_worker',WORKER_PATH)
w=importlib.util.module_from_spec(spec);spec.loader.exec_module(w)
net=w.BetaVAE().cpu()
assert sum(p.numel() for p in net.parameters()) > 12_000_000
x=torch.zeros(2,4096)
with torch.no_grad():
    mu,lv=net.encode(x)
    logits=net.decode(mu)
assert mu.shape==(2,10) and lv.shape==(2,10) and logits.shape==(2,4096)
zeros=torch.zeros(2,10); logits0=torch.zeros(2,4096)
loss0,rec0,kl0=w.loss_parts(logits0,x,zeros,zeros,4)
assert torch.allclose(rec0,torch.tensor(4096*math.log(2)),atol=1e-3)
assert torch.allclose(kl0,torch.zeros(()))
mu2=torch.ones(2,10)
_,_,kl2=w.loss_parts(logits0,x,mu2,zeros,4)
assert torch.allclose(kl2,torch.tensor(5.0))
logits,mu,lv=net(x)
loss,_,_=w.loss_parts(logits,x,mu,lv,4);loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in net.parameters())
del net, x, logits, mu, lv
print('PASS: architecture, exact pixel summed BCE, analytic KL, gradient flow')

## 5. Optional T4 wall-time preflight

The **full** training uses a fixed 40 epochs no matter what this test reports. Set `RUN_PREFLIGHT=True` to run two short **scratch** benchmarks concurrently, about 25 steps per GPU plus dataset loading. Reported throughput is approximate and may differ from sustained throughput. Benchmarks never create training checkpoints or update the final model initialization. If timing suggests a limit breach, decide *before training* whether the allowed time or the pre-agreed epoch count should change; the notebook makes no hidden adjustments.

In [ ]:
RUN_PREFLIGHT = True

def launch_workers(benchmark=False):
    procs=[]
    for gpu,beta in [(0,4.0),(1,1.0)]:
        folder=OUT/f'beta_{int(beta)}'
        folder.mkdir(exist_ok=True)
        cmd=[sys.executable,str(WORKER_PATH),'--data',str(DATA_PATH),'--out',str(folder),
             '--beta',str(beta),'--seed',str(SEED),'--epochs',str(EPOCHS),'--batch',str(BATCH_SIZE)]
        if benchmark:cmd.append('--benchmark')
        logfile=open(folder/('preflight.log' if benchmark else 'training.log'),'a',buffering=1)
        env=os.environ.copy()
        env['CUDA_VISIBLE_DEVICES']=str(gpu)
        env['PYTHONUNBUFFERED']='1'
        p=subprocess.Popen(cmd,stdout=logfile,stderr=subprocess.STDOUT,env=env)
        logfile.close()
        procs.append((beta,p,folder))
    return procs

def wait_for_workers(procs):
    try:
        while any(p.poll() is None for _,p,_ in procs):
            time.sleep(30)
            for beta,p,folder in procs:
                hfile=folder/'history.json'
                if hfile.exists():
                    h=json.loads(hfile.read_text())
                    if h:print(f'β={beta:.0f} | epoch {h[-1]["epoch"]}/{EPOCHS} | BCE {h[-1]["reconstruction_bce"]:.2f} | KL {h[-1]["kl"]:.2f} | {h[-1]["epoch_seconds"]:.0f}s/epoch')
        for beta,p,folder in procs:
            if p.returncode:
                print((folder/'training.log').read_text()[-3000:] if (folder/'training.log').exists() else 'Worker failed')
                raise RuntimeError(f'Worker β={beta} exited with code {p.returncode}')
    except KeyboardInterrupt:
        print('Interrupted: check latest saved checkpoints. Stop notebook workers if necessary.')
        raise

if RUN_PREFLIGHT:
    preflight=launch_workers(benchmark=True)
    wait_for_workers(preflight)
    for beta in [4,1]:
        result=json.loads((OUT/f'beta_{beta}'/'benchmark.json').read_text())
        print(f'β={beta} benchmark: {result["seconds_per_step"]*1000:.1f} ms/step, '
              f'estimated {result["estimated_train_hours"]:.2f} training hours, '
              f'cache {result["cache_mode"]}. Add initial data load, inference and plotting time.')
else:
    print('Skipped scratch benchmark; no impact on training configuration.')

## 6. Launch fixed-epoch training simultaneously on the two GPUs

Set `RUN_TRAINING=True` for the first run. When reopening the notebook after an interrupted session, leave it `True` to resume from saved checkpoints (same seed/configuration). Set it `False` to **only load existing checkpoints** and make plots. Each worker logs per-epoch loss components and writes exact optimizer and RNG states. If you alter the seed or training recipe, **use a new output directory** rather than resuming incompatible checkpoints. Kaggle may terminate long jobs or interrupt sessions; save/download the output directory while it is available.

In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    for beta in [4,1]:
        p=OUT/f'beta_{beta}'/'last.pt'
        if p.exists():
            try: ck=torch.load(p,map_location='cpu',weights_only=False)
            except TypeError: ck=torch.load(p,map_location='cpu')
            print(f'Existing β={beta} checkpoint at epoch {ck["epoch"]}; will resume if incomplete.')
            del ck
    workers=launch_workers(benchmark=False)
    wait_for_workers(workers)
    print('Both workers exited successfully.')
else:
    print('Training skipped; existing checkpoints must be present for subsequent cells.')

## 7. Load checkpoints, audit training and quantify constraints

The analysis uses actual learned checkpoints only. It does not infer success from epoch count. A prematurely ended run is reported with its actual epoch. Reconstruction BCE and KL are displayed separately, and each checkpoint's latent posterior variances are computed over the **entire dataset**, not a convenient subset.

In [ ]:
from collections import OrderedDict
checkpoints={}; histories={}
for beta in (4,1):
    path=OUT/f'beta_{beta}'/'last.pt'
    if not path.exists():raise FileNotFoundError(f'Missing trained checkpoint for β={beta}: {path}')
    try:ck=torch.load(path,map_location='cpu',weights_only=False)
    except TypeError:ck=torch.load(path,map_location='cpu')
    assert ck['beta']==beta and ck['seed']==SEED
    checkpoints[beta]=ck
    histories[beta]=ck['history']
    print(f'β={beta}: checkpoint at epoch {ck["epoch"]}/{EPOCHS}, final minibatch objective {ck["history"][-1]["objective"]:.2f}')
    if ck['epoch']<EPOCHS:
        print('WARNING: partial-run checkpoint; do not present as 40 completed epochs.')
fig,axes=plt.subplots(1,3,figsize=(15,4))
for beta in (4,1):
    h=histories[beta]
    for ax,key,label in zip(axes,['objective','reconstruction_bce','kl'],['Objective','BCE (sum/pixels)','KL (sum/latents)']):
        ax.plot([r['epoch'] for r in h],[r[key] for r in h],label=f'β={beta}')
        ax.set_title(label);ax.set_xlabel('Epoch');ax.grid(alpha=.2)
for ax in axes:ax.legend()
plt.tight_layout();plt.savefig(OUT/'training_diagnostics.png',dpi=160);plt.show()

## 8. Full-grid posterior evaluation (no subsampling)

Compute posterior means $\mu(x)$ and mean posterior variances $\mathbb E_x[\exp\ell(x)]$ for **every** one of the 737,280 original images and each model, in canonical factor order. The row-conditioned response maps use exact factor-wise averages:

- **Position:** mean over shape, scale, rotation → a 32×32 matrix indexed by X,Y.
- **Scale:** mean over rotation, X, Y, retaining shape → 3×6.
- **Rotation:** mean over scale, X, Y, retaining shape → 3×40.

We compute each average from the complete array, without downsampling factors. Columns are ordered from **smaller to larger** mean posterior variance, as the paper labels them as ordered by learned Gaussian variance. We also display the full-data mean per-latent KL (an additional explanatory diagnostic). This ordering may differ from the paper because learned latent indices are arbitrary.

In [ ]:
# For low host memory: only one model and its 737,280 × 10 means live in RAM at a time.
# The two output NPZ files preserve every evaluated latent (float32) for independent audit.
RESULTS={}
with np.load(DATA_PATH,allow_pickle=False) as d:
    img_np=d['imgs']   # CPU array: full 737,280 × 64 × 64, compact uint8/bool
    assert img_np.shape==(737280,64,64)
    for beta in [4,1]:
        model=w.BetaVAE().to('cuda:0').eval()
        model.load_state_dict(checkpoints[beta]['model'])
        means=np.empty((737280,10),dtype=np.float32)
        var_sum=np.zeros(10,dtype=np.float64)
        kl_sum=np.zeros(10,dtype=np.float64)
        with torch.inference_mode():
            for s in range(0,len(img_np),2048):
                e=min(s+2048,len(img_np))
                x=torch.from_numpy(np.ascontiguousarray(img_np[s:e].reshape(-1,4096),dtype=np.uint8)).to('cuda:0',dtype=torch.float32)
                mu,lv=model.encode(x)
                means[s:e]=mu.cpu().numpy()
                var_sum += lv.exp().double().sum(dim=0).cpu().numpy()
                kl_sum += (0.5*(mu.square()+lv.exp()-1-lv)).double().sum(dim=0).cpu().numpy()
        assert np.isfinite(means).all()
        # Each coordinate corresponds exactly to [shape, scale, rotation, posX, posY].
        tensor=means.reshape(3,6,40,32,32,10)
        position=tensor.mean(axis=(0,1,2))        # [X,Y,latent]
        scale=tensor.mean(axis=(2,3,4))           # [shape,scale,latent]
        rotation=tensor.mean(axis=(1,3,4))        # [shape,rotation,latent]
        var=var_sum/737280
        kl=kl_sum/737280
        order=np.argsort(var,kind='stable')
        result=dict(position=position,scale=scale,rotation=rotation,
                    mean_posterior_variance=var,mean_kl=kl,order=order,all_means=means)
        RESULTS[beta]=result
        np.savez_compressed(OUT/f'beta_{beta}'/'full_grid_responses.npz',
                            position=position,scale=scale,rotation=rotation,
                            mean_posterior_variance=var,mean_kl=kl,order=order)
        print(f'β={beta}: evaluated {len(means):,} of 737,280; order:',order.tolist())
        del model
    del img_np

## 9. Generate the original latent traversals independently from learned weights

The reference is a deterministic **oval, middle scale, middle rotation, centered position** indexed by `[color=0, shape=1, scale=2, orientation=7, X=16, Y=16]` in the canonical grid (predeclared, not selected after inspecting the results). For each model, run encoder inference on that image and set all 10 dimensions to the inferred posterior means. Then overwrite **one dimension at a time** with $[+3,+1.5,0,-1.5,-3]$, as shown in the paper; decode each without sampling. Each model has 10 independent latent columns, giving 50 generated images per model. The same source image and traversal values are used for both models. We plot probability images, not sampled Bernoulli pixels.

In [ ]:
REFERENCE_FACTORS=(0,1,2,7,16,16)
ref_flat=int(np.ravel_multi_index(REFERENCE_FACTORS,(1,3,6,40,32,32)))
with np.load(DATA_PATH,allow_pickle=False) as d:
    ref_image=np.asarray(d['imgs'][ref_flat],dtype=np.float32)
print('Reference factor indices:',REFERENCE_FACTORS,'| archive row:',ref_flat)
TRAVERSAL_VALUES=np.array([3.0,1.5,0.0,-1.5,-3.0],dtype=np.float32)
for beta in [4,1]:
    model=w.BetaVAE().to('cuda:0').eval()
    model.load_state_dict(checkpoints[beta]['model'])
    with torch.inference_mode():
        ref=torch.tensor(ref_image,device='cuda:0').reshape(1,4096)
        z0=model.encode(ref)[0][0]
        zs=z0[None,:].repeat(50,1)
        for j in range(10):
            zs[j*5:(j+1)*5,j]=torch.from_numpy(TRAVERSAL_VALUES).to('cuda:0')
        decoded=torch.sigmoid(model.decode(zs)).reshape(10,5,64,64).cpu().numpy()
    RESULTS[beta]['traversals']=decoded
    np.save(OUT/f'beta_{beta}'/'latent_traversals_probabilities.npy',decoded)
    del model
    print(f'β={beta}: decoded shape {decoded.shape}; min/max probabilities {decoded.min():.4f}/{decoded.max():.4f}')

## 10. Reproduce Figure 7A and Figure 7B

**Paper figure layout:** one column per learned latent, top rows for spatial response, scale, and orientation, five traversal rows below, and latent variance at the bottom. Our panels include **all 10** latents, not only selected active latents, explicitly revealing any unused dimensions. Spatial heatmaps use the same color scale within each model; scale and orientation share a standardized y-limit within each model; each panel labels the **actual learned latent index**. These are newly computed figures, not visual copies of the original or manually inserted curves.

Colors: square red, oval green, heart blue. The original paper's figure is on p. 10 of the attached paper; include an image of it separately in the **report** if required by your submission rubric.

In [ ]:
from matplotlib import gridspec

def render_panel(beta):
    r=RESULTS[beta]
    # Shared response limits within model; do not silently normalize each latent to maximize contrast.
    all_responses=np.concatenate([r['position'].reshape(-1,10),r['scale'].reshape(-1,10),r['rotation'].reshape(-1,10)],axis=0)
    abs_max=max(float(np.max(np.abs(all_responses))),1e-6)
    fig=plt.figure(figsize=(25,23),layout='constrained')
    gs=fig.add_gridspec(nrows=9,ncols=10,height_ratios=[2.0,1.1,1.1,1.65,1.65,1.65,1.65,1.65,.8],hspace=.12,wspace=.15)
    shape_colors=['red','green','blue'] # archive: square, oval, heart (shape indices 0,1,2)
    for col,j in enumerate(r['order']):
        ax=fig.add_subplot(gs[0,col]);ax.imshow(r['position'][:,:,j].T,origin='lower',cmap='coolwarm',vmin=-abs_max,vmax=abs_max,interpolation='nearest')
        ax.set_title(f'$z_{{{j}}}$  (rank {col+1})',fontsize=12)
        ax.set_xticks([0,31]);ax.set_yticks([0,31])
        if col==0:ax.set_ylabel('Position Y',fontsize=11);ax.set_xlabel('Position X',fontsize=9)
        else:ax.set_yticklabels([])
        ax=fig.add_subplot(gs[1,col]);
        for s,color in enumerate(shape_colors):ax.plot(np.arange(6),r['scale'][s,:,j],c=color,lw=1.6)
        ax.set_xlim(0,5);ax.set_ylim(-abs_max,abs_max);ax.grid(alpha=.2)
        if col==0:ax.set_ylabel('Mean response\nscale',fontsize=10)
        else:ax.set_yticklabels([])
        ax=fig.add_subplot(gs[2,col]);
        for s,color in enumerate(shape_colors):ax.plot(np.arange(40)*2*np.pi/40,r['rotation'][s,:,j],c=color,lw=1.4)
        ax.set_xlim(0,2*np.pi);ax.set_ylim(-abs_max,abs_max);ax.set_xticks([0,np.pi,2*np.pi],['0','π','2π']);ax.grid(alpha=.2)
        if col==0:ax.set_ylabel('Mean response\nrotation',fontsize=10)
        else:ax.set_yticklabels([])
        for k,v in enumerate(TRAVERSAL_VALUES):
            ax=fig.add_subplot(gs[3+k,col]);ax.imshow(r['traversals'][j,k],vmin=0,vmax=1,cmap='gray',interpolation='nearest')
            ax.set_xticks([]);ax.set_yticks([])
            if col==0:ax.set_ylabel(f'$z_j={v:g}$',fontsize=12)
        ax=fig.add_subplot(gs[8,col]);ax.axis('off')
        ax.text(.5,.7,f'var = {r["mean_posterior_variance"][j]:.3g}',ha='center',va='center',fontsize=9)
        ax.text(.5,.24,f'KL = {r["mean_kl"][j]:.3g}',ha='center',va='center',fontsize=9)
    fig.suptitle(f'Figure 7{"A" if beta==4 else "B"} reproduction — β={beta} | all 10 learned latents, variance-ordered | {checkpoints[beta]["epoch"]} epochs',fontsize=17)
    path=OUT/f'Figure_7{"A" if beta==4 else "B"}_beta{beta}.png'
    fig.savefig(path,dpi=125,bbox_inches='tight')
    plt.show();plt.close(fig)
    print('Saved:',path)
    return path

FIGURE_PATHS={beta:render_panel(beta) for beta in (4,1)}

## 11. Additional diagnostic: actual reconstructions and per-latent capacity

A side-by-side depiction of reconstruction probabilities against the reference source image, plus per-latent mean KL and learned posterior variance, helps assess whether a seemingly disentangled response is due to true factor learning or simply an inactive/collapsed code. The reference reconstruction is not one of the traversals unless it coincides with a traversal coordinate. Interpret neither mean KL nor mean Gaussian variance as a standalone disentanglement metric.

In [ ]:
fig,axs=plt.subplots(2,4,figsize=(15,7))
for row,beta in enumerate((4,1)):
    r=RESULTS[beta]
    model=w.BetaVAE().to('cuda:0').eval()
    model.load_state_dict(checkpoints[beta]['model'])
    with torch.inference_mode():
        source=torch.tensor(ref_image,device='cuda:0').reshape(1,4096)
        recon=torch.sigmoid(model.decode(model.encode(source)[0])).cpu().numpy().reshape(64,64)
    axs[row,0].imshow(ref_image,cmap='gray',vmin=0,vmax=1);axs[row,0].set_title('Ground truth image')
    axs[row,1].imshow(recon,cmap='gray',vmin=0,vmax=1);axs[row,1].set_title(f'Reconstruction β={beta}')
    axs[row,2].bar(np.arange(10),r['mean_kl']);axs[row,2].set_title('Mean per-latent KL')
    axs[row,3].bar(np.arange(10),r['mean_posterior_variance']);axs[row,3].set_title('Mean posterior variance')
    for j in range(4):
        if j<2:axs[row,j].axis('off')
        else:axs[row,j].set_xlabel('Latent index')
    del model
plt.tight_layout();plt.savefig(OUT/'reconstruction_and_capacity_diagnostics.png',dpi=160);plt.show()

## 12. Interpretation, limitations, and Phase 3 report checklist

**What the comparison tests:** In a well-disentangled run, individual posterior-mean response curves should depend strongly on one ground-truth factor and only weakly on others, and the corresponding decoder traversals should modify a coherent single visual attribute. β=4 is expected to show a cleaner separation than β=1 *in the original experiment*; our run must be interpreted from actual plots, and a different outcome is possible.

**One-seed limitation:** With one seed per β we can establish whether these two specific trained models reproduce the claimed qualitative comparison. We **cannot** infer seed-to-seed robustness, confidence intervals, or the average benefit of β=4. We do not select the seed producing the most attractive plot. Latent indices need not match the paper.

**Key ambiguities documented:** Batch size, epoch count, initialization, Adagrad epsilon, and Figure 7's reference image are not specified by the paper. Here they are explicitly fixed; the 40-epoch training target is a compute-budget choice. Decoder's final layer emits Bernoulli logits; tanh is used only in hidden layers. Learned variance is defined as $\mathbb E_x\sigma_j^2(x)$, not $\operatorname{Var}_x\mu_j(x)$. The two quantities must not be conflated. We preserve all 10 latents; the paper presents a smaller selection including unused units. Factor labels are only used after training.

**Failure analysis if outputs differ:** Check the verified factor ordering, BCE pixel sum, optimizer settings, training curves and plateaus, mean KL and posterior variances, reference reconstructions, and whether both runs completed 40 epochs. An apparently flat response can indicate a genuinely unused latent, but also an undertrained model. A latent that captures multiple factors shows responses/traversals with coupled variation. Record observed evidence before speculating about paper-specific missing hyperparameters.

**Submission steps** (Assignment §1.1 item 2 and §4):

1. Include the paper's *original Figure 7* alongside these reproduced Figure 7A and 7B images in your report; cite the paper appropriately.
2. Explain mathematics and architecture, seed count, learned-variance ordering, and the fixed traversal protocol, and discuss changes in disentanglement and reconstruction.
3. Provide the share link for the complete LLM conversation and identify the LLM name/version as required by the assignment.
4. Keep a dated record of issues encountered in running this notebook, fixes, and any residual mismatch; do **not** invent successes or failures.
5. Commit this **Phase 3** independently developed notebook and its instructions **before** comparing any existing authors' code. Do not consult existing implementations until permitted Phase 4; make the GitHub repository public **on the submission date**, per the assignment.

**End-to-end provenance:** Input is only the original dSprites archive. Both checkpoints, histories, summary response arrays, latent traversals, figures, training logs and scratch preflight benchmarks are saved under `OUT`. A first run without an attached dataset requires Kaggle Internet; later runs can reuse attached data and saved checkpoints. Results require executing the GPU training cells — no outcomes are simulated in this distributed notebook.